# v1.3 全画面手检测：MobileNetV1 骨干实验

依据厂商 MobileNetV1 示例和支持的 Conv2D/DepthwiseConv2D 算子。采用ImageNet预训练前五个深度可分离块，加手框预测头；这不是厂商预训练手势检测模型。输入256×144灰度，输出32×18网格。独立验证未达标时禁止声称部署完成。

In [1]:
# v1.3: project-only Jupyter execution; no board programming.
import os,sys,json,hashlib
from pathlib import Path
import numpy as np
import tensorflow as tf
ROOT=Path.cwd()
while not (ROOT/'AGENTS.md').exists():
    if ROOT==ROOT.parent: raise RuntimeError('Open this notebook inside the CNN project')
    ROOT=ROOT.parent
assert ROOT.name=='fpga-w.-codex-cnn'
sys.path.insert(0,str(ROOT/'ml/tinyml'))
from workflow import export_int8,predict_tflite,digest
os.environ['KERAS_HOME']=str(ROOT/'ml/tinyml/keras')
tf.keras.utils.set_random_seed(1301)
tf.config.threading.set_intra_op_parallelism_threads(6)
tf.config.threading.set_inter_op_parallelism_threads(2)
print('TensorFlow',tf.__version__,'GPU',tf.config.list_physical_devices('GPU'))


TensorFlow 2.16.2 GPU []


In [2]:
sys.path.insert(0,str(ROOT/'ml/detector'))
from train import data,encode,augment,detector_loss
from export import stats
from workflow import split_detection_heads
training=data('train');validation=data('val')
out=ROOT/'ml/tinyml/artifacts/detector';out.mkdir(parents=True,exist_ok=True)
print('Train/validation',len(training),len(validation))
base=tf.keras.applications.MobileNet(input_shape=(224,224,3),alpha=.25,include_top=False,weights='imagenet')
front=tf.keras.Model(base.input,base.get_layer('conv_pw_5_relu').output)
inputs=tf.keras.Input((144,256,1),name='gray')
backbone=tf.keras.models.clone_model(front,input_tensors=inputs)
for layer in backbone.layers:
    if not layer.weights:continue
    weights=front.get_layer(layer.name).get_weights()
    if layer.name=='conv1':weights[0]=weights[0].sum(axis=2,keepdims=True)
    layer.set_weights(weights)
# Repeated-gray RGB input is exactly equivalent to summing conv1 RGB kernels.
x=tf.keras.layers.Conv2D(32,3,padding='same',activation='relu',name='hand_context')(backbone.output)
head=tf.keras.layers.Conv2D(5,1,bias_initializer=tf.keras.initializers.Constant([-2.19,.5,.5,.15,.25]),name='hand_head')(x)
model=tf.keras.Model(inputs,head,name='mobilenetv1_hand_detector')
assert model.output_shape==(None,18,32,5)
model.summary()


Train/validation 5284 628


      0/2108140 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

  16384/2108140 ━━━━━━━━━━━━━━━━━━━━ 10s 5us/step

  24576/2108140 ━━━━━━━━━━━━━━━━━━━━ 11s 5us/step

  32768/2108140 ━━━━━━━━━━━━━━━━━━━━ 11s 6us/step

...

Model: "mobilenetv1_hand_detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gray (InputLayer)               │ (None, 144, 256, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 72, 128, 8)     │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_bn (BatchNormalization)   │ (None, 72, 128, 8)     │            32 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_relu (ReLU)               │ (None, 72, 128, 8)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1 (DepthwiseConv2D)     │ (None, 72, 128, 8)     │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_bn                    │ (None, 72, 128, 8)     │            32 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_1_relu (ReLU)           │ (None, 72, 128, 8)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1 (Conv2D)              │ (None, 72, 128, 16)    │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_bn                    │ (None, 72, 128, 16)    │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_1_relu (ReLU)           │ (None, 72, 128, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pad_2 (ZeroPadding2D)      │ (None, 73, 129, 16)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2 (DepthwiseConv2D)     │ (None, 36, 64, 16)     │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_bn                    │ (None, 36, 64, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_2_relu (ReLU)           │ (None, 36, 64, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2 (Conv2D)              │ (None, 36, 64, 32)     │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_bn                    │ (None, 36, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_2_relu (ReLU)           │ (None, 36, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3 (DepthwiseConv2D)     │ (None, 36, 64, 32)     │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_bn                    │ (None, 36, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_dw_3_relu (ReLU)           │ (None, 36, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3 (Conv2D)              │ (None, 36, 64, 32)     │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_pw_3_bn                    │ (None, 36, 64, 32)     │           128 │
│ (BatchNormalization)            │                        │             

 Total params: 29,349 (114.64 KB)

 Trainable params: 28,613 (111.77 KB)

 Non-trainable params: 736 (2.88 KB)

## 训练
前3轮冻结预训练骨干，随后27轮微调；按人员划分，平移/缩放/翻转同时变换手框。校准/阈值选择不读取测试集。

In [ ]:
rng=np.random.default_rng(1301)
def generator():
    while True:
        for i in rng.permutation(len(training)):
            im,b,_=training[i];im,b=augment(im,b,rng)
            yield ((im.astype(np.float32)-128)/128)[...,None],encode(b)
options=tf.data.Options();options.threading.private_threadpool_size=4
ds=tf.data.Dataset.from_generator(generator,output_signature=(tf.TensorSpec((144,256,1),tf.float32),tf.TensorSpec((18,32,6),tf.float32))).batch(48).prefetch(2).with_options(options)
vx=np.stack([r[0] for r in validation])[...,None].astype(np.float32)
vy=np.stack([encode(r[1]) for r in validation])
vd=tf.data.Dataset.from_tensor_slices(((vx-128)/128,vy)).batch(48).with_options(options)
for layer in backbone.layers:layer.trainable=False
model.compile(tf.keras.optimizers.Adam(.001),loss=detector_loss)
model.fit(ds,steps_per_epoch=(len(training)+47)//48,validation_data=vd,epochs=3,verbose=2)
for layer in backbone.layers:layer.trainable=True
model.compile(tf.keras.optimizers.Adam(.0003),loss=detector_loss)
history=model.fit(ds,steps_per_epoch=(len(training)+47)//48,validation_data=vd,epochs=27,verbose=2,callbacks=[tf.keras.callbacks.ModelCheckpoint(str(out/'best.keras'),monitor='val_loss',save_best_only=True),tf.keras.callbacks.CSVLogger(str(out/'training.csv')),tf.keras.callbacks.ReduceLROnPlateau(patience=6,factor=.5,min_lr=1e-5)])
model=tf.keras.models.load_model(out/'best.keras',compile=False)


## 独立INT8检测头
分离object logit和坐标的输出量化尺度；两头浮点输出应与训练时的5通道输出完全一致。原先固定Q8的RTL不能直接执行这里的TFLite模型。

In [ ]:
deployment=split_detection_heads(model)
p=model.predict((vx[:8]-128)/128,verbose=0)
a,b=deployment.predict((vx[:8]-128)/128,verbose=0)
assert np.max(np.abs(np.concatenate([a,b],axis=-1)-p))<1e-5
indices=np.random.default_rng(1301).choice(len(training),480,replace=False)
calibration=(np.stack([training[i][0] for i in indices])[...,None].astype(np.float32)-128)/128
path,contract=export_int8(deployment,calibration,out,'hand_detector_int8')
print(json.dumps(contract,indent=2))


In [ ]:
outputs=predict_tflite(path,(vx-128)/128)
score=next(x for x in outputs if x.shape[-1]==1);boxes=next(x for x in outputs if x.shape[-1]==4)
q=np.floor(np.concatenate([score,boxes],axis=-1)*256+.5).astype(np.int32)
f=np.floor(model.predict((vx-128)/128,batch_size=48,verbose=0)*256+.5).astype(np.int32)
thresholds=[-768,-512,-384,-256,-128,0,128,256,512,768]
results=[stats(q,validation,t) for t in thresholds]
eligible=[r for r in results if r['precision_iou50']>=.85 and r['scene_recall_iou50']>=.50]
report=dict(version='v1.3',architecture='MobileNetV1 alpha .25 blocks1..5, gray conv1 folded, context conv32, separate INT8 heads',epochs=30,model_sha256=digest(out/'best.keras'),tflite_sha256=digest(path),validation=results,float_validation=[stats(f,validation,t) for t in thresholds],deployment_gate_pass=bool(eligible),selected=max(eligible,key=lambda r:r['scene_recall_iou50']) if eligible else None,test_evaluated=False,hardware_verified=False)
(out/'validation_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))


## 下一步
门槛未通过：继续训练/诊断，保留实验TFLite用于算子兼容性检查，不发布部署结果。门槛通过：冻结阈值后再运行未使用的测试集和全流程定位+分类评估，再做目标TFLite Micro及硬件验证。